In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import math
import json

with open('neo4j_dbinfo', 'r') as f:
    neo4j_info = json.load(f)


uri = neo4j_info["uri"]
user = neo4j_info["username"]
password = neo4j_info["password"]

driver = GraphDatabase.driver(
    uri,
    auth=(user, password)
)

CSV_PATH = "cpdat_v4.0_product_composition_data.csv"
df = pd.read_csv(CSV_PATH)


def clean_value(v):

    if pd.isna(v):
        return None

    if isinstance(v, float) and math.isnan(v):
        return None

    return v


QUERY = """
CREATE (n:CPDAT {

    data_source: $data_source,
    data_source_url: $data_source_url,

    data_group: $data_group,
    data_group_url: $data_group_url,
    data_group_download_date: $data_group_download_date,

    data_document_type: $data_document_type,
    cpdat_data_document_id: $cpdat_data_document_id,

    data_document_title: $data_document_title,
    data_document_subtitle: $data_document_subtitle,

    data_document_url: $data_document_url,
    data_document_date: $data_document_date,

    organization: $organization,

    cpdat_product_id: $cpdat_product_id,
    product_title: $product_title,
    product_upc: $product_upc,

    puc_kind: $puc_kind,
    puc_general_category: $puc_general_category,
    puc_product_family: $puc_product_family,
    puc_product_type: $puc_product_type,
    puc_classification_method: $puc_classification_method,

    raw_chemical_name: $raw_chemical_name,
    raw_casrn: $raw_casrn,
    cpdat_raw_chemical_id: $cpdat_raw_chemical_id,

    dtxsid: $dtxsid,

    curated_chemical_name: $curated_chemical_name,
    curated_casrn: $curated_casrn,

    provisional_dtxsid: $provisional_dtxsid,

    raw_min_comp: $raw_min_comp,
    raw_max_comp: $raw_max_comp,
    raw_central_comp: $raw_central_comp,

    composition_units: $composition_units,

    lower_weight_fraction: $lower_weight_fraction,
    upper_weight_fraction: $upper_weight_fraction,
    central_weight_fraction: $central_weight_fraction,

    weight_fraction_type: $weight_fraction_type,

    component_name: $component_name
})
"""

with driver.session() as session:

    total = len(df)

    for idx, row in df.iterrows():

        params = {
            col: clean_value(row[col])
            for col in df.columns
        }

        session.run(QUERY, params)

        if idx % 100 == 0:
            print(f"{idx}/{total} inserted")


driver.close()

print("DONE")